# Hong Kong Property Data · cloud client

Install the Python package once, then read the latest published dataset for your own analysis. The cloud performs the source collection. This notebook reads data into memory without running a crawler or creating local data files. Python 3.10+ and internet access are required; pandas, NumPy, Git and ChatGPT are not required.

[Dashboard](https://hogan-cyber.github.io/Real-Estate-Data/) · [Package guide](https://github.com/Hogan-cyber/Real-Estate-Data/blob/main/PYTHON_PACKAGE.md)

## 1. Install the package

Run once or when upgrading code. If an older package version was already imported, restart the kernel after installation, then continue below. No numerical libraries are installed or upgraded by the package.

In [ ]:
%pip install --quiet --upgrade "https://hogan-cyber.github.io/Real-Estate-Data/packages/hk_property_data-0.2.0-py3-none-any.whl"

## 2. Read the published dataset

The complete snapshot includes historical data and can exceed 50 MB and is transferred over the network into memory. `fetch_dataset()` returns a dictionary; tables are lists of dictionaries. Read frequency and source_period: C01 nominal GDP is quarterly. The publisher's timestamp and original source warnings are preserved. The printed timestamp identifies this snapshot, not the time you ran the notebook.

In [ ]:
from hk_property_data import fetch_dataset, __version__

data = fetch_dataset()
observations = data["observations"]
rates = data["rate_observations"]

print("Package version:", __version__)
print("Published source-check time:", data["checked_at"])
print("Monthly records:", len(observations))
print("Daily/effective rates:", len(rates))
print("Source issues:", data["errors"] or "None")
for note in data.get("warnings", []):
    print("Note:", note)

## 3. Use time-series observations

This example selects the original Hong Kong dollar M3 series. Values remain in their published units; missing values remain `None`. Change the metric ID to select a different series.

In [ ]:
METRIC_ID = "hk_m3_hkd"
selected = [row for row in observations if row["metric_id"] == METRIC_ID]

for row in sorted(selected, key=lambda row: row["date"])[-5:]:
    print(row["period"], row["metric_name"], row["value"], row["unit"])

## 4. Use daily and effective rates

These are separate from official monthly averages. HIBOR dates are fixing dates; Prime dates are effective dates of rate changes. An old effective date can still describe the current rate. The history available for each series can differ.

In [ ]:
latest_rates = {}
for row in sorted(rates, key=lambda row: row["date"]):
    latest_rates[row["metric_id"]] = row

for row in latest_rates.values():
    print(row["date"], row["metric_name"], row["value"], row["unit"])

hibor_1m = [row for row in rates if row["metric_id"] == "hk_hibor_1m_daily"]
print("1M daily records available:", len(hibor_1m))

## 5. Optional DataFrames

If pandas already works in your environment, run the following in your own cell. It is optional; all preceding cells work without pandas or NumPy. This notebook does not install or upgrade them.

```python
import pandas as pd
monthly_df = pd.DataFrame(monthly)
rates_df = pd.DataFrame(rates)
```

## 6. Future updates and Excel

The cloud updates independently on its existing daily GitHub Actions schedule. Rerun **step 2** to read the latest published data into your variables; reinstalling the package is unnecessary when data changes. Use `fetch_status()` if you only need to check the cloud publication time. A returned dictionary stays unchanged until you fetch again.

[Open/download the published Excel workbook](https://hogan-cyber.github.io/Real-Estate-Data/data/hk_property_data.xlsx) · [Database documentation](https://github.com/Hogan-cyber/Real-Estate-Data/blob/main/DATABASE.md)

Excel downloads are optional. The package's cloud read does not save a dataset, Excel file, database or cache on your computer. Network failures raise an error; source-refresh issues remain visible in `data["errors"]` and `data["source_status"]`. This is access to a published snapshot, not a request to rerun the official-source crawler.

## 7. Sales and bank mortgage data

Amounts are HKD million, not per-flat prices. Counts measure registered agreements, loans, or all-property deeds as stated in each row. Bank purchase approvals = primary + secondary approvals; no non-bank residual is calculated. Each series can end in a different month.


In [ ]:
metric_ids = [
    "hk_residential_sales_count",
    "hk_residential_primary_sales_count",
    "hk_residential_secondary_sales_count",
    "hk_mortgage_approved_count",
    "hk_mortgage_drawn_count",
    "hk_mortgage_approved_refinancing_count",
    "hk_mortgage_approved_purchase_count",
    "hk_all_property_mortgage_deeds_count",
]
for metric_id in metric_ids:
    history = sorted([r for r in observations if r["metric_id"] == metric_id and r["value"] is not None], key=lambda r: r["period"])
    if history:
        row = history[-1]
        print(metric_id, history[0]["period"], row["period"], row["value"], row["unit"], row.get("observation_type", "official"))
    else:
        print(metric_id, "Unavailable in this published snapshot")


In [ ]:
purchase_approvals = [r for r in observations if r["metric_id"] == "hk_mortgage_approved_purchase_count" and r["period"] >= "2026-01"]
for row in purchase_approvals:
    print(row["period"], row["value"], row["unit"], row["derivation"], row["source_url"])


Optional pandas view, only if pandas already works in your environment:

```python
import pandas as pd
df = pd.DataFrame(observations)
sales_and_mortgages = df[df["metric_id"].isin(metric_ids)]
sales_and_mortgages.pivot(index="period", columns="metric_id", values="value").tail(12)
```

Do not fill unavailable months with zero. Rerun `data = fetch_dataset()` and `observations = data["observations"]` for a later snapshot.
